# 03 · Answers to the brief's three prompts

**Purpose.** Answer the brief's three example prompts directly from the matching code, with no AI involved: the
numbers the agent's answers must reproduce. How each prompt is read, and the results for every marketed listing.

The agent (Claude with the tools) gives these same answers in conversation; the evals check that it does
(DESIGN.md section 5).


## 1. Prompt 1: the most similar properties

**Purpose.** State how we answer the brief's first example prompt, and (once `similarity.py` is complete) show and
check the answers for each marketed listing.

The evidence behind each choice is in [`02_similarity`](02_similarity.ipynb).

### 1.1 The question

> "Find the 10 most similar properties in the estate to the newly marketed property."

How we read it:

* **The input is one marketed listing.** Each listing (M1-M5) is a single unit, e.g. M3 is Spitfire Park, Unit 42.
* **The answer is 10 units from Indurent's portfolio** ("the estate"): the 1,250 units most like that listing.
* **"Properties" means units, and the answer is the literal top 10: no cap on units per estate.** Units on the
  estate a listing sits on genuinely are among the most similar (same location, often similar age and size), and
  every listing with coordinates sits on an existing estate (`02_similarity`, section 2). So a top 10 can be concentrated: in a
  first rough ranking, all 10 of M3's are on Velvet Park 60, its own estate. The answer says so when it happens
  (*"all 10 are on Velvet Park 60, the estate M3 sits on"*).
* **A cap is available on request.** If the user asks for a spread ("similar units on other estates", "at most 2
  per estate"), the ranking can limit units per estate. It's off by default because it changes the answer away
  from "the 10 most similar". (EDA section 4.3 first suggested a cap; this replaces that suggestion.)
* **"The newly marketed property" is singular, but there are five listings.** The answer either asks which one is
  meant or answers for all five; it never picks one silently.

### 1.2 How similarity is defined

Each portfolio unit gets **three separate similarity scores** against the listing, one per block, each between 0
and 1. They use the same form, `0.5 ** (gap / half-life)`: 1 for an identical value, halving with every half-life of
difference, and the same whichever direction the difference goes.

| Block | Compares | 1 when | Halves every | Evidence |
|---|---|---|---|---|
| **Age** | build years | built the same year | 10 years apart (roughly one era of building specification) | `02_similarity` section 1 |
| **Location** | coordinates (straight-line miles) | in the same spot | 10 miles apart (a local industrial market) | `02_similarity` section 2 |
| **Physical** | size (sq ft, log scale) | the same size | each doubling (or halving) in size | `02_similarity` section 3 |

**Why size uses a ratio, while age and location use a fixed gap.** Each half-life is measured in the kind of
difference that matters for that quantity:

* **Location and age: an absolute gap.** 10 miles apart means much the same anywhere, and 10 years between 1975 and
  1985 is about as big a step in building specification as 10 years between 2000 and 2010.
* **Size: a relative gap.** The same 1,000 sq ft difference means very different things: 1,000 vs 2,000 sq ft is
  twice the size (a different class of unit), while 100,000 vs 101,000 sq ft is 1% bigger (effectively the same).
  A fixed sq ft half-life would get one of these badly wrong; a ratio treats 1k vs 2k the same as 50k vs 100k, each
  a step up to a different class of building. It's the same reason size is plotted on a log scale.

**Physical means size only.** Size is the only physical field every listing has; eaves exist for M4 and M5 only, and
usable door counts for 39 portfolio units. Eaves, doors and volume are shown as context, not scored (`02_similarity`, section 3).

**Combining the blocks into one ranking:**

* **Overall score = the average of the three blocks, equally weighted** by default; the weights can be changed.
* **A block the listing doesn't have is dropped** for that listing, and the answer says so: M2 has no build year or
  coordinates, so it's matched on size only; M5 is offshore, so it's matched on age and size.
* **A block a portfolio unit doesn't have scores 0**, so a unit we know less about can't outrank one that matches on
  everything. 358 units have no usable location and 16 no build year; they can still appear, marked
  "unknown" on that block.
* **No cap on units per estate** by default; an optional cap if the user asks for a spread.
* **All three block scores are kept**, not just the average, so an answer can say *why* a unit is similar, e.g. "on
  the same estate and a similar size, but 25 years newer".

Which blocks each listing can be matched on:

In [1]:
from pathlib import Path

import pandas as pd

from indurent_matching.data import load_marketed, load_portfolio

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
portfolio = load_portfolio(ROOT / "data" / "Current Portfolio.xlsx")
marketed = load_marketed(ROOT / "data" / "Marketed Warehouses.xlsx")

blocks = pd.DataFrame({
    "age": marketed["build_year"].notna() & ~marketed["build_year_doubtful"],
    "location": marketed["location_confidence"] != "unknown",
    "physical": marketed["size_sqft"].notna() & ~marketed["size_sqft_doubtful"],
}, index=marketed.index)
blocks.insert(0, "listing", marketed["industrial_estate_name"] + ", " + marketed["unit_name"])
blocks["matched on"] = blocks[["age", "location", "physical"]].apply(
    lambda row: ", ".join(block for block, ok in row.items() if ok), axis=1)
blocks.replace({True: "yes", False: "-"})

,listing,age,location,physical,matched on
listing_id,,,,,
M1,"Cherry Lane, Unit 10",yes,yes,yes,"age, location, physical"
M2,"Tech Hub, Unit 8",-,-,yes,physical
M3,"Spitfire Park, Unit 42",yes,yes,yes,"age, location, physical"
M4,"Stable Lane, Unit 14",yes,yes,yes,"age, location, physical"
M5,"Chancery Depot, Unit 2",yes,-,yes,"age, physical"


In [2]:
# The same check on the portfolio side: units missing a block score 0 on it.
pd.Series({
    "no usable build year": (portfolio["build_year"].isna() | portfolio["build_year_doubtful"]).sum(),
    "no usable location": (portfolio["location_confidence"] == "unknown").sum(),
    "no usable size": (portfolio["size_sqft"].isna() | portfolio["size_sqft_doubtful"]).sum(),
}, name="portfolio units").to_frame()

,portfolio units
no usable build year,16
no usable location,358
no usable size,10


* **M1, M3 and M4** can be matched on all three blocks.
* **M2 (Tech Hub)** only on physical (size): it has no build year and no coordinates.
* **M5 (Chancery Depot)** on age and physical: its coordinates are 27 km offshore.
* On the portfolio side, most units have a usable build year and size; **358 have no usable location**, so they
  score 0 on location when a listing is matched on it.

### 1.3 The top 10 for each listing

`most_similar` from `similarity.py`, with the defaults: equal weights, no cap on units per estate. Block scores are
shown blank where the unit has no usable value (it counted 0 in `overall`) or where the listing can't be compared
on that block.

In [3]:
from contextlib import contextmanager

import indurent_matching.similarity as similarity
from indurent_matching.similarity import most_similar


SHOW = ["industrial_estate_name", "unit_name", "overall", "age", "location", "physical",
        "build_year", "size_sqft", "miles_from_listing"]
top10 = {listing_id: most_similar(row, portfolio) for listing_id, row in marketed.iterrows()}


def show(listing_id):
    row = marketed.loc[listing_id]
    print(f"{listing_id} {row['industrial_estate_name']}, {row['unit_name']}: built {row['build_year']:.0f}, "
          f"{row['size_sqft']:,.0f} sq ft, matched on {blocks.loc[listing_id, 'matched on']}"
          if pd.notna(row["build_year"]) else
          f"{listing_id} {row['industrial_estate_name']}, {row['unit_name']}: {row['size_sqft']:,.0f} sq ft, "
          f"matched on {blocks.loc[listing_id, 'matched on']}")
    return top10[listing_id][SHOW].round({"overall": 2, "age": 2, "location": 2, "physical": 2,
                                          "size_sqft": 0, "miles_from_listing": 1})


show("M1")

M1 Cherry Lane, Unit 10: built 1977, 16,860 sq ft, matched on age, location, physical


,industrial_estate_name,unit_name,overall,age,location,physical,build_year,size_sqft,miles_from_listing
property_id,,,,,,,,,
586,Quartz Wharf 16,Unit 5,0.88,0.93,0.73,0.97,1976.0,17340.0,4.5
584,Quartz Wharf 16,Unit 3,0.81,0.87,0.74,0.82,1975.0,20565.0,4.4
1157,Scarlet Pavilion 78,Unit 7,0.76,0.76,1.00,0.52,1981.0,8752.0,0.1
606,Quartz Wharf 16,Unit 25,0.72,0.47,0.74,0.96,1988.0,16208.0,4.4
612,Quartz Wharf 16,Unit 31,0.69,0.76,0.73,0.57,1981.0,9603.0,4.5
642,Quartz Wharf 16,Unit 61,0.67,0.76,0.73,0.53,1973.0,8945.0,4.5
673,Ruby Junction 67,Unit 7,0.65,1.00,NaN,0.95,1977.0,17793.0,NaN
1153,Scarlet Pavilion 78,Unit 3,0.64,0.81,1.00,0.12,1974.0,2011.0,0.0
418,Verdant Oaks 7,Unit 9,0.64,1.00,NaN,0.91,1977.0,15273.0,NaN


In [4]:
show("M2")

M2 Tech Hub, Unit 8: 906 sq ft, matched on physical


,industrial_estate_name,unit_name,overall,age,location,physical,build_year,size_sqft,miles_from_listing
property_id,,,,,,,,,
602,Quartz Wharf 16,Unit 21,1.00,NaN,NaN,1.00,1979.0,905.0,NaN
689,Ruby Junction 67,Unit 23,1.00,NaN,NaN,1.00,2006.0,905.0,NaN
221,Opal Hub 4,Unit 2,1.00,NaN,NaN,1.00,2022.0,905.0,NaN
874,Velvet Park 60,Unit 29,1.00,NaN,NaN,1.00,2004.0,902.0,NaN
698,Ruby Junction 67,Unit 32,1.00,NaN,NaN,1.00,1998.0,901.0,NaN
24,Sable Mill 91,Unit 6,0.99,NaN,NaN,0.99,2016.0,898.0,NaN
1101,Stone Mews 81,Unit 4,0.98,NaN,NaN,0.98,1986.0,889.0,NaN
601,Quartz Wharf 16,Unit 20,0.98,NaN,NaN,0.98,2017.0,923.0,NaN
863,Velvet Park 60,Unit 18,0.98,NaN,NaN,0.98,1976.0,923.0,NaN


In [5]:
show("M3")

M3 Spitfire Park, Unit 42: built 1976, 2,294 sq ft, matched on age, location, physical


,industrial_estate_name,unit_name,overall,age,location,physical,build_year,size_sqft,miles_from_listing
property_id,,,,,,,,,
869,Velvet Park 60,Unit 24,0.84,0.81,0.99,0.71,1973.0,1638.0,0.1
865,Velvet Park 60,Unit 20,0.81,0.76,1.00,0.69,1980.0,3332.0,0.0
863,Velvet Park 60,Unit 18,0.80,1.00,1.00,0.40,1976.0,923.0,0.1
855,Velvet Park 60,Unit 10,0.80,0.87,1.00,0.52,1978.0,1199.0,0.1
878,Velvet Park 60,Unit 33,0.77,0.66,0.99,0.66,1970.0,1520.0,0.1
849,Velvet Park 60,Unit 4,0.77,0.44,1.00,0.88,1988.0,2013.0,0.0
851,Velvet Park 60,Unit 6,0.74,0.93,0.99,0.29,1977.0,675.0,0.1
860,Velvet Park 60,Unit 15,0.73,0.76,0.99,0.45,1980.0,5086.0,0.1
852,Velvet Park 60,Unit 7,0.73,0.41,1.00,0.79,1989.0,2904.0,0.0


In [6]:
show("M4")

M4 Stable Lane, Unit 14: built 2000, 617 sq ft, matched on age, location, physical


,industrial_estate_name,unit_name,overall,age,location,physical,build_year,size_sqft,miles_from_listing
property_id,,,,,,,,,
26,Sable Mill 91,Unit 8,0.70,0.54,1.00,0.56,2009.0,1098.0,0.0
27,Sable Mill 91,Unit 9,0.69,0.76,0.99,0.31,2004.0,1985.0,0.1
25,Sable Mill 91,Unit 7,0.68,0.22,0.99,0.84,2022.0,734.0,0.1
24,Sable Mill 91,Unit 6,0.67,0.33,1.00,0.69,2016.0,898.0,0.0
20,Sable Mill 91,Unit 2,0.66,0.93,1.00,0.05,1999.0,11839.0,0.0
871,Velvet Park 60,Unit 26,0.66,1.00,0.00,0.97,2000.0,639.0,118.1
88,Scarlet Grove 78,Unit 7,0.65,0.93,0.09,0.93,2001.0,667.0,34.8
21,Sable Mill 91,Unit 3,0.64,0.76,1.00,0.18,2004.0,3479.0,0.1
844,Ruby Arc 96,Unit 47,0.62,0.87,0.00,0.99,2002.0,613.0,116.1


In [7]:
show("M5")

M5 Chancery Depot, Unit 2: built 1966, 16,614 sq ft, matched on age, physical


,industrial_estate_name,unit_name,overall,age,location,physical,build_year,size_sqft,miles_from_listing
property_id,,,,,,,,,
482,Sapphire Grove 50,Unit 10,0.93,1.00,NaN,0.87,1966.0,14412.0,NaN
478,Sapphire Grove 50,Unit 6,0.87,0.76,NaN,0.97,1962.0,17052.0,NaN
810,Ruby Arc 96,Unit 13,0.84,0.71,NaN,0.98,1971.0,17017.0,NaN
258,Steel Junction 81,Unit 4,0.81,0.71,NaN,0.91,1971.0,15141.0,NaN
656,Onyx Beacon 83,Unit 4,0.80,0.93,NaN,0.67,1965.0,11101.0,NaN
918,Crimson Mill 42,Unit 1,0.78,0.71,NaN,0.86,1971.0,19332.0,NaN
211,Scarlet Depot 92,Unit 7,0.78,0.76,NaN,0.81,1970.0,13429.0,NaN
53,Ruby Mill 13,Unit 4,0.78,0.76,NaN,0.80,1970.0,20683.0,NaN
210,Scarlet Depot 92,Unit 6,0.78,1.00,NaN,0.56,1966.0,9221.0,NaN


**What the top 10s look like:**

* **M1 (Cherry Lane, Wales, 1977, 16,860 sq ft)**: led by **Quartz Wharf 16**, the estate 4.4 miles away, where
  several units match closely on age and size (Unit 5: 1976, 17,340 sq ft, score 0.88). Two units on Scarlet
  Pavilion 78, the estate M1 sits on, also appear. Three of the ten have no usable location (blank), but match
  on age and size almost exactly.
* **M2 (Tech Hub, 906 sq ft)**: matched on size alone, so the top 10 are simply the units closest to 906 sq ft,
  built anywhere between 1976 and 2022.
* **M3 (Spitfire Park, Midlands, 1976, 2,294 sq ft)**: all ten are on **Velvet Park 60**, the estate it sits on. They
  score about 1 on location, and the order within them is set by age and size.
* **M4 (Stable Lane, 2000, 617 sq ft)**: seven on **Sable Mill 91**, its own estate, plus three tiny units built
  around 2000 elsewhere, two of them over 100 miles away. With equal weights, a unit that matches perfectly on age
  and size can rank in the top 10 from anywhere (location 0, but 1 + 1 out of 3 = 0.67).
* **M5 (Chancery Depot, offshore, 1966, 16,614 sq ft)**: matched on age and size only, so the top 10 are mid-sized
  units built 1962-1971, from eight estates around the country.

### 1.4 Checks on the ranking

**Where the top 10 come from, and which block they score highest on.** For each listing: how many of its top 10
are on the estate it sits on, how many estates they span, and the average of each block across the top 10.

In [8]:
from indurent_matching.similarity import sits_on_estate

summary = pd.DataFrame({
    listing_id: {
        "sits on": sits_on_estate(marketed.loc[listing_id], portfolio),
        "top 10 on that estate": (t["industrial_estate_name"] == sits_on_estate(marketed.loc[listing_id], portfolio)).sum(),
        "estates in top 10": t["industrial_estate_name"].nunique(),
        "mean overall": t["overall"].mean(),
        "mean age": t["age"].mean(),
        "mean location": t["location"].mean(),
        "mean physical": t["physical"].mean(),
        "location unknown in top 10": t["location"].isna().sum() if blocks.loc[listing_id, "location"] else None,
    }
    for listing_id, t in top10.items()
}).T
summary.round(2)

,sits on,top 10 on that estate,estates in top 10,mean overall,mean age,mean location,mean physical,location unknown in top 10
M1,Scarlet Pavilion 78,2,5,0.708976,0.835593,0.809553,0.724649,3
M2,NaN,0.0,7.0,0.990428,NaN,NaN,0.990428,NaN
M3,Velvet Park 60,10,1,0.772911,0.716859,0.996157,0.605716,0
M4,Sable Mill 91,7,4,0.65912,0.687162,0.706468,0.583731,0
M5,NaN,0.0,8.0,0.815316,0.803503,NaN,0.827129,NaN


* **Location dominates when a listing sits on an estate**: M3's ten are all on its own estate, M4's mostly. That's
  the expected result of no cap (section 1.1): units on the same estate really are the closest comparables.
* **M1 is different**: its own estate (Scarlet Pavilion 78) is small and its units differ in size from M1, so a
  nearby bigger estate (Quartz Wharf 16) supplies most of the matches.
* **Unknown locations do appear** (three of M1's ten). They score 0 on location, so they need near-perfect age and
  size to get in, and an answer should flag them as "location unknown".

**Sensitivity to the half-lives.** Each half-life is halved and doubled in turn, and the table shows how many
of each listing's default top 10 survive. 10 means the choice doesn't matter for that listing; a low number means
the ranking is sensitive to it.

In [9]:
HALF_LIVES = {"age": "AGE_HALF_LIFE_YEARS", "location": "LOCATION_HALF_LIFE_MILES", "physical": "SIZE_HALF_LIFE_RATIO"}


@contextmanager
def half_life(block, factor):
    """Temporarily scale one block's half-life (for size, the ratio is raised to the factor: 2 -> 4 or 1.41)."""
    name = HALF_LIVES[block]
    original = getattr(similarity, name)
    setattr(similarity, name, original**factor if block == "physical" else original * factor)
    try:
        yield
    finally:
        setattr(similarity, name, original)


rows = {}
for block in HALF_LIVES:
    for factor, label in [(0.5, "halved"), (2, "doubled")]:
        with half_life(block, factor):
            rows[f"{block} {label}"] = {
                listing_id: len(set(most_similar(row, portfolio).index) & set(top10[listing_id].index))
                for listing_id, row in marketed.iterrows()
            }
pd.DataFrame(rows).T

,M1,M2,M3,M4,M5
age halved,9,10,8,9,8
age doubled,8,10,8,9,8
location halved,9,10,10,10,10
location doubled,8,10,8,8,10
physical halved,9,10,8,9,8
physical doubled,7,10,9,9,8


* **The rankings are fairly stable**: halving or doubling any one half-life keeps 7-10 of a listing's 10 matches.
  So the exact choices (10 years, 10 miles, twice the size) shape the results but don't decide them.
* **Location's half-life matters most when it's doubled.** Halving it (5 miles) changes almost nothing: M3's ten
  stay on Velvet Park 60. Doubling it (20 miles) swaps two units each for M1, M3 and M4, as estates 15-25 miles
  away start to count: M3 gains one unit each on Granite Pointe 39 and Cedar Pavilion 55. M5 isn't matched on
  location at all.
* **M2 never changes**: with size alone, the order of units doesn't depend on how fast similarity falls off.

**The optional cap.** M3's top 10 with at most 2 units per estate, for when the user asks for a spread:

In [10]:
most_similar(marketed.loc["M3"], portfolio, max_per_estate=2)[SHOW].round(
    {"overall": 2, "age": 2, "location": 2, "physical": 2, "size_sqft": 0, "miles_from_listing": 1})

,industrial_estate_name,unit_name,overall,age,location,physical,build_year,size_sqft,miles_from_listing
property_id,,,,,,,,,
869,Velvet Park 60,Unit 24,0.84,0.81,0.99,0.71,1973.0,1638.0,0.1
865,Velvet Park 60,Unit 20,0.81,0.76,1.00,0.69,1980.0,3332.0,0.0
713,Sapphire Oaks 10,Unit 15,0.67,1.00,NaN,1.00,1976.0,2299.0,NaN
465,Granite Pointe 39,Unit 31,0.66,0.76,0.23,0.99,1980.0,2323.0,21.4
299,Cedar Pavilion 55,Unit 5,0.65,0.87,0.29,0.80,1978.0,1828.0,17.8
1059,Ruby Bridge 60,Unit 4,0.64,0.93,0.00,1.00,1977.0,2295.0,179.5
87,Scarlet Grove 78,Unit 6,0.64,0.93,0.00,0.97,1977.0,2355.0,152.8
710,Sapphire Oaks 10,Unit 12,0.63,0.93,NaN,0.97,1977.0,2219.0,NaN
769,Crimson Forge 9,Unit 9,0.63,0.93,0.00,0.94,1975.0,2167.0,96.0


* With the cap, M3's list keeps its two best Velvet Park 60 units, then spreads out: units built 1975-1980 of
  almost exactly M3's size, on estates 18-180 miles away or with no usable location.
* That's the "show me similar units on other estates" answer: useful on request, but not the literal top 10.

**M2 is matched on size alone, so many units tie.** How many portfolio units score as well as M2's 10th
match (to two decimal places)?

In [11]:
m2_all = most_similar(marketed.loc["M2"], portfolio, k=len(portfolio))
tenth = m2_all["overall"].iloc[9]
print(f"M2's 10th match scores {tenth:.3f}; {(m2_all['overall'].round(2) >= round(tenth, 2)).sum()} units score "
      f"{round(tenth, 2):.2f} or more")

M2's 10th match scores 0.979; 11 units score 0.98 or more


* **11 units score 0.98 or more for M2**, so which ten make its list comes down to tiny size differences (and
  Property Id for exact ties). The answer should say M2 is matched on size only, and that many units are about as
  close.

### 1.5 Summary

* `most_similar` returns sensible comparables: on the estate a listing sits on when its units match, from nearby
  estates when they don't (M1), and on age and size alone when there's no usable location (M5) or only size (M2).
* The ranking is reasonably stable to the half-life choices.
* Things answers should state: which blocks a listing was matched on; when the top 10 is concentrated on one
  estate; units with an unknown location; and, for size-only matches, that many units tie.
* **Weights stay equal.** Equal weights let a unit 100+ miles away into the top 10 if its age and size match
  perfectly (M4's #6 and #9), but there's no evidence in the data for favouring one block, and the brief asks for
  all three dimensions. A user who cares more about location can say so, and the weights change for that answer
  (`weights={"location": 2}`); the answer should point such distant units out.

## 2. Prompt 2: homogeneity scores

**Purpose.** State how we answer the brief's second example prompt, and show the answers for each marketed listing.

The evidence behind each score is in [`02_similarity`](02_similarity.ipynb).

### 2.1 The question

> "Provide a correlation score for the homogeneity of the marketed property(ies) with the rest of the estate,
> scoring each by physical characteristics, location, and age separately."

How we read it:

* **"Correlation score" is ambiguous.** Strictly, correlation is between two variables, not between one property and
  a portfolio. We read it as a **homogeneity score: how typical the listing is of Indurent's portfolio**, and every
  answer states that interpretation.
* **One score per dimension, never combined**: age, location and physical, as the prompt asks.
* **Each score runs from 0 to 1, scaled so the most typical value in the portfolio scores 1.** 0.5 means about half
  as typical as the most typical; near 0 means unusual for the portfolio.
* **"Not available" is not 0.** A listing with no usable value for a dimension gets no score for it, and the answer
  says why.
* **Each score comes with context** that explains it but isn't scored.

### 2.2 How each score is defined

| Dimension | Score | Context alongside |
|---|---|---|
| **Age** | Portfolio units in the listing's 5-year build-year band (1975-79, ...), divided by the units in the most frequent band | |
| **Location** | The best-placed portfolio estate's mean miles to its 5 nearest other estates, divided by the listing's mean miles to its 5 nearest estates (leaving out the estate it sits on) | The estate the listing sits on; the nearest other estate and its distance; the share of portfolio estates it's better placed than |
| **Physical** | Portfolio units of a similar size to the listing (size / 1.41 to size x 1.41), divided by the most units any portfolio unit has of a similar size | Eaves height against similar-sized units ("tall for its size") |

Why these forms, in short (details in `02_similarity`):

* **Age** uses fixed bands because build years are flat from 1970 to 2021; a moving window would let the 2022 spike
  set the scale.
* **Location** uses estates, not units, as the neighbours, because every listing with coordinates sits on an existing
  Indurent estate: its nearest units would all be on that estate, at about 0 miles.
* **Physical** means size only, the one physical field every listing has; it uses a moving range because sizes slope
  steadily and fixed bands made artificial jumps.

In [12]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

from indurent_matching.data import load_marketed, load_portfolio
from indurent_matching.similarity import homogeneity_scores

pio.renderers.default = "plotly_mimetype+notebook_connected"

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
portfolio = load_portfolio(ROOT / "data" / "Current Portfolio.xlsx")
marketed = load_marketed(ROOT / "data" / "Marketed Warehouses.xlsx")

results = {listing_id: homogeneity_scores(row, portfolio) for listing_id, row in marketed.iterrows()}

### 2.3 The scores

`homogeneity_scores` from `similarity.py` for each listing:

In [13]:
scores = pd.DataFrame(
    {listing_id: {block: r[block] for block in ["age", "location", "physical"]} for listing_id, r in results.items()}
).T
scores.insert(0, "listing", marketed["industrial_estate_name"] + ", " + marketed["unit_name"])
scores.round(2).astype(object).fillna("not available")

,listing,age,location,physical
M1,"Cherry Lane, Unit 10",0.86,0.23,0.41
M2,"Tech Hub, Unit 8",not available,not available,0.38
M3,"Spitfire Park, Unit 42",0.86,0.62,0.87
M4,"Stable Lane, Unit 14",0.86,0.73,0.17
M5,"Chancery Depot, Unit 2",0.2,not available,0.41


In [14]:
BLOCK_COLOURS = {"age": "#2a78d6", "location": "#eb6834", "physical": "#1baf7a"}  # validated categorical slots 1-3
listings = list(scores.index)  # M1 to M5, left to right

fig = go.Figure()
for block, colour in BLOCK_COLOURS.items():
    values = scores.loc[listings, block]
    fig.add_bar(
        x=listings, y=values.fillna(0), name=block, marker_color=colour, marker_cornerradius=4,
        text=[f"{v:.2f}" if pd.notna(v) else "n/a" for v in values],  # n/a: not available
        textposition="outside", textangle=0, textfont_color="#55554f", cliponaxis=False,
        hovertemplate=f"{block}: %{{text}}<extra>%{{x}}</extra>",
    )
fig.update_layout(
    title="Homogeneity with the portfolio, by dimension (1 = most typical)",
    barmode="group", bargap=0.25, bargroupgap=0.08, height=460, template="plotly_white",
    yaxis={"range": [0, 1.15], "tickvals": [0, 0.25, 0.5, 0.75, 1], "title": "Score"},
    xaxis={"title": None},
    legend={"orientation": "h", "y": 1.1, "x": 0, "traceorder": "normal"},
    uniformtext={"minsize": 9, "mode": "show"},
)
fig.show()

* **Age**: M1, M3 and M4 (1976-2000) are in the portfolio's main run of build years and score 0.86 each. **M5 (1966)
  scores 0.20**: Indurent owns little stock from before 1970. M2 has no build year.
* **Location**: **M4 (0.73) and M3 (0.62)** are in well-populated parts of the portfolio; **M1 (0.23)** is in Wales,
  where Indurent owns little beyond the estate it sits on. M2 has no coordinates and M5's are 27 km offshore.
* **Physical**: **M3 (0.87)** is close to the portfolio's most common size; **M4 (617 sq ft, 0.17)** is smaller than
  99% of the portfolio. Every listing has a size, so every listing has a physical score.
* So no listing is typical on every dimension: **M3 comes closest** (0.86 / 0.62 / 0.87), M4 is typical on age and
  location but unusually small, and M1 is typical on age but isolated.

**The context behind each score:**

In [15]:
pd.DataFrame({listing_id: r["location_context"] for listing_id, r in results.items()}).T.infer_objects().round(1)

,sits_on_estate,nearest_other_estate,nearest_other_estate_miles,better_placed_than_pct
M1,Scarlet Pavilion 78,Quartz Wharf 16,4.4,6.6
M2,NaN,NaN,NaN,NaN
M3,Velvet Park 60,Jade Bridge 6,9.1,62.9
M4,Sable Mill 91,Sable Beacon 12,6.3,85.4
M5,NaN,NaN,NaN,NaN


In [16]:
pd.DataFrame({listing_id: r["size_context"] for listing_id, r in results.items()}).T.infer_objects().round(1)

,eaves_m,similar_size_units_with_eaves,their_median_eaves_m,eaves_higher_than_pct
M1,NaN,NaN,NaN,NaN
M2,NaN,NaN,NaN,NaN
M3,NaN,NaN,NaN,NaN
M4,7.0,6.0,4.5,83.3
M5,5.3,45.0,6.3,15.6


* **M1, M3 and M4 each sit on an estate Indurent already owns** (Scarlet Pavilion 78, Velvet Park 60, Sable Mill
  91). That's the most useful single location fact, and it leads the answer even when the score is low (M1).
* M1's nearest other estate is only 4.4 miles away, but its 5 nearest average 36 miles: hence 0.23, better placed than
  only 7% of Indurent's estates.
* **Eaves context exists only for M4 and M5.** M4's 7 m eaves are tall for its size (higher than 83% of 6 similar
  units); M5's 5.3 m are a little low (higher than 16% of 45).

### 2.4 What an answer looks like

The numbers come from the code; the model turns them into prose, stating the interpretation, the scores, the context
and what's missing. For M3, for example:

> *Homogeneity here means how typical Spitfire Park, Unit 42 is of Indurent's portfolio, scored 0-1 where 1 is the
> most typical. **Age 0.86**: built in 1976, within the portfolio's main run of build years. **Location 0.62**: it
> sits on Velvet Park 60, an estate Indurent already owns; the next nearest estate is Jade Bridge 6, 9 miles away,
> and it's better placed than 63% of Indurent's estates. **Physical 0.87**: at 2,294 sq ft it's close to the
> portfolio's most common size (no eaves height recorded).*

And for M2, where most is missing:

> *Tech Hub, Unit 8 can only be scored on physical characteristics: it has no build year and no coordinates, so age
> and location are not available. **Physical 0.38**: at 906 sq ft it's smaller than 95% of Indurent's units.*

### 2.5 Summary

* Every listing gets three separate scores (or "not available"), each 0-1 with the most typical value in the portfolio
  scoring 1, and each with context that explains it.
* The answers should always state: the interpretation of "correlation score", which dimensions are missing and why,
  and, for M1, M3 and M4, the estate the listing sits on.

## 3. Prompt 3: the closest properties near a major city

> "Find the closest properties to the marketed property, after excluding any property in the portfolio that is
> more than 10 miles from a major city."

How we read it:

* **"Closest" means nearest on the map**, by straight-line miles: a different question from prompt 1's "most
  similar".
* **"Excluding any property more than 10 miles from a major city" means keep units with `miles_to_city <= 10`**,
  using the 16 major cities in `config.py`.
* **Units with no usable location are left out** (their distance to a city is unknown), and the answer says how
  many, offering to include them.
* A listing that can't be placed on the map (M2: no coordinates; M5: offshore) has no nearest properties.

Two steps, using the tools the agent calls (`src/indurent_matching/tools.py`): `filter_rows` to keep the units
within 10 miles of a city, then `find_nearest` on that set.


In [17]:
import json
from pathlib import Path

import pandas as pd

from indurent_matching.data import load_marketed, load_portfolio
from indurent_matching.tools import (
    Data,
    ToolError,
    filter_rows,
    find_nearest,
    find_similar,
    get_homogeneity_scores,
    get_listing,
    list_listings,
)


ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
data = Data(load_portfolio(ROOT / "data" / "Current Portfolio.xlsx"),
            load_marketed(ROOT / "data" / "Marketed Warehouses.xlsx"))


def show(result):
    """Print a tool result as the model receives it."""
    print(json.dumps(result, indent=2))

In [18]:
within_10 = [{"field": "miles_to_city", "op": "<=", "value": 10}]
excluded = filter_rows(data, within_10, include_unknown=False)
kept = filter_rows(data, within_10, include_unknown=True)
print(excluded["summary"])
print(kept["summary"])
print(f"\ncandidates (first 10 of {len(excluded['candidates'])}): {excluded['candidates'][:10]}")

Kept 50 of 1,250 units (miles_to_city <= 10). 358 had no usable value for miles_to_city and were excluded. (miles_to_city is unknown where a unit has no usable location.)
Kept 408 of 1,250 units (miles_to_city <= 10). 358 had no usable value for miles_to_city and were kept. (miles_to_city is unknown where a unit has no usable location.)

candidates (first 10 of 50): [167, 168, 169, 170, 275, 276, 277, 278, 336, 337]


* **50 of 1,250** units are known to be within 10 miles of a major city. **358 have no usable location**: excluded,
  the comparison is against 50 units; kept, against 408. The summary always reports the 358, so the answer can say
  which choice was made.
* The result's `candidates` (Property Ids) are the new set; the agent remembers them and passes them to the next
  tool.


### 3.1 The nearest units to M3

Prompt 3 asks for the **closest** properties, which we read as **nearest on the map**: a different question from
prompt 1's "most similar". `find_nearest` ranks units by straight-line miles. Prompt 3 end to end is two calls:
`filter_rows` (above) to keep the 50 units within 10 miles of a major city, then `find_nearest` on that set:

In [19]:
result = find_nearest(data, "M3", candidates=excluded["candidates"])
print(result["summary"])
for caveat in result["caveats"]:
    print("-", caveat)
pd.DataFrame(result["matches"]).set_index("property_id")

The 10 units nearest to M3 (Spitfire Park, Unit 42) on the map, from 50 units with a usable location
- 12 units are 68.8 miles or nearer (to 1 decimal place), so these 10 aren't the only ones that near.
- The listing sits on Velvet Park 60, an estate Indurent already owns (at least one of its units is within 0.25 miles of the listing); 0 of the 10 nearest are on it. None of Velvet Park 60's units are in the filtered set: the filter removed them.
- Searched a filtered set of 50 units, not the whole portfolio.


,estate,unit,miles_from_listing,build_year,size_sqft,nearest_city,miles_to_city
property_id,,,,,,,
170,Jade Bridge 6,Unit 4,9.1,1989,7908,Manchester,7.1
169,Jade Bridge 6,Unit 3,9.1,1958,17520,Manchester,7.1
167,Jade Bridge 6,Unit 1,9.1,1977,7542,Manchester,7.1
168,Jade Bridge 6,Unit 2,9.2,1968,8239,Manchester,7.1
503,Scarlet Arc 73,Unit 1,16.2,2005,134579,Sheffield,9.0
1162,Topaz Harbour 22,Unit 1,51.5,2004,15268,Nottingham,8.2
483,Onyx Oaks 5,Unit 1,62.7,1987,173913,Leicester,9.0
348,Marble Works 29,Unit 1,64.8,1987,887038,Birmingham,7.9
401,Sapphire Junction 78,Unit 4,68.8,2000,38568,Birmingham,8.7


* **M3's nearest unit within 10 miles of a city is Jade Bridge 6, 9.1 miles away.** None of Velvet Park 60, the
  estate M3 sits on, survives the filter: it's 14 miles from Manchester, and the caveat says "0 of the 10 nearest
  are on it".
* The caveats record that the search was over a filtered set of 50, so the answer can say so (and, from the
  filter's summary, that 358 units with no usable location were left out).

A listing that can't be placed on the map has no nearest properties; the error says why, for the answer to pass
on:

### 3.2 The other listings

The same for M1 and M4, the other listings that can be placed on the map:


In [20]:
for listing in ["M1", "M4"]:
    result = find_nearest(data, listing, k=5, candidates=excluded["candidates"])
    print(result["summary"])
    for caveat in result["caveats"]:
        print("-", caveat)
    display(pd.DataFrame(result["matches"]).set_index("property_id"))


The 5 units nearest to M1 (Cherry Lane, Unit 10) on the map, from 50 units with a usable location
- The listing sits on Scarlet Pavilion 78, an estate Indurent already owns (at least one of its units is within 0.25 miles of the listing); 0 of the 5 nearest are on it. None of Scarlet Pavilion 78's units are in the filtered set: the filter removed them.
- Searched a filtered set of 50 units, not the whole portfolio.


,estate,unit,miles_from_listing,build_year,size_sqft,nearest_city,miles_to_city
property_id,,,,,,,
348,Marble Works 29,Unit 1,74.6,1987,887038,Birmingham,7.9
400,Sapphire Junction 78,Unit 3,88.1,2020,27960,Birmingham,8.6
399,Sapphire Junction 78,Unit 2,88.1,2015,42332,Birmingham,8.6
398,Sapphire Junction 78,Unit 1,88.1,1987,31721,Birmingham,8.6
401,Sapphire Junction 78,Unit 4,88.1,2000,38568,Birmingham,8.7


The 5 units nearest to M4 (Stable Lane, Unit 14) on the map, from 50 units with a usable location
- 8 units are 39.9 miles or nearer (to 1 decimal place), so these 5 aren't the only ones that near.
- The listing sits on Sable Mill 91, an estate Indurent already owns (at least one of its units is within 0.25 miles of the listing); 0 of the 5 nearest are on it. None of Sable Mill 91's units are in the filtered set: the filter removed them.
- Searched a filtered set of 50 units, not the whole portfolio.


,estate,unit,miles_from_listing,build_year,size_sqft,nearest_city,miles_to_city
property_id,,,,,,,
350,Onyx Pointe 12,Unit 1,33.2,NaN,284391,London,8.6
894,Sable Crescent 13,Unit 1,39.9,1992.0,8053,London,6.9
899,Sable Crescent 13,Unit 6,39.9,2005.0,14818,London,6.9
896,Sable Crescent 13,Unit 3,39.9,2013.0,8391,London,6.9
900,Sable Crescent 13,Unit 7,39.9,1977.0,4712,London,6.9


In [21]:
for listing in ["M2", "M5"]:
    try:
        find_nearest(data, listing)
    except ToolError as error:
        print(f"find_nearest({listing!r}) -> ToolError: {error}")

find_nearest('M2') -> ToolError: M2 can't be placed on the map: no coordinates. Nearest properties can't be found for it; say so in the answer.
find_nearest('M5') -> ToolError: M5 can't be placed on the map: coordinates are offshore, so can't be trusted. Nearest properties can't be found for it; say so in the answer.
